# Prompt Preview Notebook

This notebook allows you to preview the prompts generated for different stages of the marketing intelligence pipeline: **Analysis**, **Recommendation**, and **Evaluation**.

In [14]:
import sys
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Add src to path
current_dir = Path(os.getcwd())
project_root = current_dir.parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / '.env')

from src.shared.utils.prompt_loader import PromptLoader
from src.shared.utils.prompt_builder import PromptBuilder
from src.shared.utils.prompt_registry import PromptRegistry

print("Environment setup complete.")

Environment setup complete.


## 1. Setup Sample Data
We'll define some sample data to populate the prompts.

In [15]:
campaign_data = {
    "campaign_id": 1,
    "platform": "Google Ads",
    "spend": 2662.38,
    "revenue": 4803.43,
    "conversions": 159,
    "cpa": 16.74
}

business_domain = {
    "industry": "Retail",
    "offering": "Membership Plan for essential shoppers"
}

campaign_target = {
    "goal": "Traffic",
    "audience": "First-time investors"
}

analysis_report = {
    "analysis": {
        "executive_summary": "The campaign is misaligned with the target audience...",
        "budget_and_efficiency": [],
        "results_and_value": [],
        "cross_channel_patterns_and_risks": [],
        "channel_notes": []
    }
}

recommendation = {
    "title": "Optimize Audience Targeting",
    "suggestion": "Shift targeting from investors to essential shoppers.",
    "priority": "High"
}

loader = PromptLoader.from_module_dir()
builder = PromptBuilder(loader=loader)

# NEW: Goal-specific instructions loading
primary_goal = campaign_target.get("goal")
goal_prompt_path = PromptRegistry.get_objective_prompt(primary_goal)
goal_instructions = ""
if goal_prompt_path:
    goal_instructions = loader.load_prompt_text(goal_prompt_path)

print(f"Sample data and builder initialized. Goal: {primary_goal}")
print(f"Goal Instructions loaded: {len(goal_instructions)} characters.")

Sample data and builder initialized. Goal: Traffic
Goal Instructions loaded: 794 characters.


## 2. Preview Analysis Prompt

In [16]:
analysis_messages = builder.build_analysis_prompt(
    campaign_data=campaign_data,
    business_domain=business_domain,
    campaign_target=campaign_target,
    goal_instructions=goal_instructions
)

print("=== SYSTEM PROMPT ===\n")
print(analysis_messages[0]['content'])
print("\n=== USER PROMPT ===\n")
print(analysis_messages[1]['content'])

print("\n--- Verification ---")
if "{{GOAL_INSTRUCTIONS}}" not in analysis_messages[0]['content']:
    print("SUCCESS: {{GOAL_INSTRUCTIONS}} placeholder replaced.")
else:
    print("FAILURE: {{GOAL_INSTRUCTIONS}} placeholder still present.")

=== SYSTEM PROMPT ===

Persona
You are a Digital Campaign Senior Analyst with strong business judgment.
You translate campaign performance data into clear business meaning for non-marketers.

Instruction
Analyze the provided campaign data to explain:
- Budget efficiency (how well spend is used)
- Business outcomes (leads, revenue, conversions)
- Patterns, risks, and gaps

You must produce structured analytical insights only.
DO NOT provide recommendations.

Context
This analysis will be used by business stakeholders to understand campaign performance.
They are not marketing experts.
STRATEGIC GOAL: Traffic
Objective: Drive a high volume of high-intent visitors to the platform at the lowest sustainable cost.

PRIORITY METRICS: clicks, ctr, cpc.
SECONDARY SIGNALS: landing_page_views, cpm, bounce_proxy_rate, session_quality_score.

ANALYSIS GUIDELINES:
1. Intent Analysis: Prioritize high CTR over low CPC. A low CPC is useless if the traffic doesn't engage with the content.
2. Cost Managem

## 3. Preview Recommendation Prompt

In [17]:
rec_messages = builder.build_recommendation_prompt(
    campaign_target=campaign_target,
    business_domain=business_domain,
    campaign_data=campaign_data,
    analysis_json=analysis_report,
    goal_instructions=goal_instructions
)

print("=== SYSTEM PROMPT ===\n")
print(rec_messages[0]['content'])
print("=== USER PROMPT ===\n")
print(rec_messages[1]['content'])

print("\n--- Verification ---")
if "{{GOAL_INSTRUCTIONS}}" not in rec_messages[0]['content']:
    print("SUCCESS: {{GOAL_INSTRUCTIONS}} placeholder replaced.")
else:
    print("FAILURE: {{GOAL_INSTRUCTIONS}} placeholder still present.")

=== SYSTEM PROMPT ===

Persona
You are a Digital Campaign Performance Strategist.
You turn campaign analysis into clear, actionable business decisions.

Instruction
Based on the provided analysis, generate EXACTLY 4 actionable recommendations.

Each recommendation must:
- Address a real issue from the analysis
- Be specific and implementable
- Be prioritized by business impact

Context
The output will be used by business stakeholders to take action.
STRATEGIC GOAL: Traffic
Objective: Drive a high volume of high-intent visitors to the platform at the lowest sustainable cost.

PRIORITY METRICS: clicks, ctr, cpc.
SECONDARY SIGNALS: landing_page_views, cpm, bounce_proxy_rate, session_quality_score.

ANALYSIS GUIDELINES:
1. Intent Analysis: Prioritize high CTR over low CPC. A low CPC is useless if the traffic doesn't engage with the content.
2. Cost Management: Identify platforms where CPC is rising without a corresponding increase in CTR or Page Views.
3. Traffic Quality: Watch the "Landin

## 4. Preview Evaluation Prompt (Analysis)

In [18]:
eval_analysis_messages = builder.build_analysis_evaluation_prompt(
    campaign_data=campaign_data,
    analysis_report=analysis_report,
    campaign_target=campaign_target,
    business_domain=business_domain
)
print("=== SYSTEM PROMPT ===\n")
print(eval_analysis_messages[0]['content'])
print("=== USER PROMPT ===\n")
print(eval_analysis_messages[1]['content'])

=== SYSTEM PROMPT ===

You are a Senior Digital Marketing Analyst and Quality Auditor. 
Your task is to evaluate the quality of a generated campaign analysis report.

Criteria for Evaluation:
1. Clarity: Is the analysis easy to understand and specifically tied to the provided metrics? (Score 1-3)
2. Accuracy: Does the analysis logically follow from the raw campaign data? Does it avoid hallucinations? (Score 1-3)
3. Structure: Does the analysis follow the required sections (Executive Summary, Budget & Efficiency, Results & Value, Risks, Channel Notes)? (Score 1-3)

Evaluation Format:
- Always provide reasoning for each score.
- Provide a final verdict: 'accepted', 'revise', or 'reject'.
- List specific key issues and improvement suggestions.

Strictly adhere to the provided JSON schema. Ensure your output is a JSON object with the following structure:
{
  "evaluation": {
    "clarity_score": <number 1-3>,
    "clarity_reasoning": "<string>",
    "accuracy_score": <number 1-3>,
    "accu

## 5. Preview Evaluation Prompt (Recommendation)

In [19]:
eval_rec_messages = builder.build_recommendation_evaluation_prompt(
    business_domain=business_domain,
    campaign_target=campaign_target,
    campaign_data=campaign_data,
    analysis_context=analysis_report,
    recommendation=recommendation
)
print("=== SYSTEM PROMPT ===\n")
print(eval_rec_messages[0]['content'])
print("=== USER PROMPT ===\n")
print(eval_rec_messages[1]['content'])

=== SYSTEM PROMPT ===

You are a Senior Digital Marketing Auditor.
Your task is to evaluate the quality of marketing recommendation cards generated for a non-marketer end user.

Criteria for Evaluation:
1. Clarity: Is the recommendation actionable and easy to understand? (Score 1-3)
2. Accuracy: Is it logically grounded in the provided analysis? (Score 1-3)
3. Structure: Does it follow the standardized card format (Title, What's Happening, What to Do, Why it Matters, Priority, Impact)? (Score 1-3)
4. Feasibility: Is it realistic given platform capabilities? (Score 1-3)

Evaluation Format:
- Provide detailed reasoning for each score.
- Provide a final verdict: 'accept', 'revise', or 'reject'.
- List critical issues and suggestions for improvement.

Strictly adhere to the provided JSON schema. Ensure your output is a JSON object with the following structure:
{
  "evaluation": {
    "clarity_score": <number 1-3>,
    "clarity_reasoning": "<string>",
    "accuracy_score": <number 1-3>,
   